## Understanding Cardinality — What Does a Row Represent?

**Cardinality** describes the grain of your dataset: what one row corresponds to in the real world.

Getting this wrong leads to double-counting, broken joins, and inflated metrics. Before writing any query, always ask:

> *"What uniquely identifies a single row in this table?"*

Common examples:

| Grain | One row = | Typical identifying columns |
| --- | --- | --- |
| **One row per pupil** | A single pupil | `pupil_id` |
| **One row per pupil per school** | A pupil's enrolment at a specific school | `pupil_id`, `school_urn` |
| **One row per pupil per school per term** | A pupil's enrolment in a given term | `pupil_id`, `school_urn`, `term` |

The more columns needed to identify a row, the **finer** the grain. Aggregating to a coarser grain (e.g. from pupil-term to pupil) requires explicit decisions about how to summarise — do you take the latest term, sum across terms, or pick the mode?

### How to check cardinality

The simplest test: count total rows versus distinct combinations of the columns you believe form the grain. If the numbers differ, you have duplicates at that grain.

In [0]:
%sql
-- Example: a small pupil-school dataset
-- Each row should represent one pupil at one school

CREATE OR REPLACE TEMP VIEW pupil_school AS
SELECT * FROM VALUES
  (1, 'Alice',  100, 'Oak Academy'),
  (2, 'Bob',    100, 'Oak Academy'),
  (3, 'Carol',  200, 'Elm School'),
  (1, 'Alice',  200, 'Elm School'),     -- Alice moved schools: valid at pupil+school grain
  (2, 'Bob',    100, 'Oak Academy')     -- Bob appears twice at the same school: duplicate!
AS t(pupil_id, pupil_name, school_urn, school_name);

SELECT * FROM pupil_school;

In [0]:
%sql
-- Cardinality check: compare total rows to distinct key combinations
SELECT
  COUNT(*)                                    AS total_rows,
  COUNT(DISTINCT concat(pupil_id, '-', school_urn)) AS distinct_pupil_school,
  CASE
    WHEN COUNT(*) = COUNT(DISTINCT concat(pupil_id, '-', school_urn))
    THEN 'No duplicates at pupil+school grain'
    ELSE 'Duplicates exist at pupil+school grain'
  END AS cardinality_check
FROM pupil_school;